<font color="green"><h2> **Welcome to ABT/HYD 182**


 ## **Lab 9**: GeoAI – Instance Segmentation (Mask R-CNN)


Replace each **XXXXX** in the code cells with the correct variable or value. Markdown cells above each code block explain what to enter and where.

### **Due Date: March 11 | 11:59 PM | 2026**
------------------------------------------------------------------------------

## Academic Integrity Statement

**This work was completed without the use of Generative AI tools (such as ChatGPT, Copilot, etc.).**

By completing the information below, you certify that you have completed this assignment independently and without the assistance of generative AI tools. This lab is designed to help you learn GeoAI and instance segmentation through hands-on practice.

---

**Note:** If you did use a generative AI tool, you must clearly disclose this in your notebook, including which tool you used and how you used it (e.g., debugging, understanding error messages, or clarifying concepts). Failure to disclose the use of generative AI tools may be considered a violation of course academic integrity policies.

In [ ]:
# Enter your information below
your_name = ""  # Replace with your full name
date = ""       # Replace with today's date (e.g., "March 5, 2026")

# Print your statement
print("Academic Integrity Statement")
print("=" * 50)
print(f"Name: {your_name}")
print(f"Date: {date}")
print("=" * 50)
print("I certify that this work was completed without the use of Generative AI tools.")

## **Table of Contents**

[Setup](#section_setup) – Install packages, mount Drive & create lab folder  
1. [Exercise 1](#section1) – Download sample data  
2. [Exercise 2](#section2) – Visualize sample data  
3. [Exercise 3](#section3) – Create training data  
4. [Exercise 4](#section4) – Train instance segmentation model  
5. [Exercise 5](#section5) – Run inference  
6. [Exercise 6](#section6) – Vectorize masks & add geometric properties  
7. [Exercise 7](#section7) – Visualize results on test set (box plot, filter by area)  
8. [Exercise 8](#section8) – Compare predictions on train set (actual vs predicted, metrics)  
9. [Exercise 9](#section9) – Model performance  

--------------------------------------------
Learning objectives
---------------------------------------------

* In this lab you will:

    *   install and use the **geoai** package for geospatial AI
    *   download sample imagery and vector labels for building detection
    *   create training tiles and train a **Mask R-CNN** instance segmentation model
    *   run inference, vectorize masks, and add geometric properties
    *   visualize and compare predictions with imagery
    *   interpret training metrics and understand instance vs semantic segmentation

This notebook is based on training instance segmentation models for object detection (e.g., building detection) using Mask R-CNN. Unlike semantic segmentation, instance segmentation distinguishes between individual objects of the same class.

Resources:
- [GeoAI documentation](https://opengeoai.org/)
- [OpenGeos GeoAI examples](https://github.com/opengeos/geoai)

<a name="section_setup"></a>
## **Setup** – Get ready for the lab

Run the cells below **in order** once before starting the exercises:

1. **Check GPU** – This lab needs a T4 GPU; set it in Runtime → Change runtime type if you haven't already.
2. **Install the GeoAI package** – Install the `geoai-py` package.
3. **Import the packages** – Import geoai and all other packages used in this lab (leafmap, numpy, matplotlib, etc.).
4. **Mount Google Drive** – Mount Drive and create the lab folder so all data and outputs are saved under **My Drive → ABT182_GeoAI**.
5. **Colab Pro (Yes/No)** – Tell the notebook whether you use Colab Pro or free tier so it can set batch sizes and memory options.

---


### 1. Check GPU (required)

This lab runs on GPU. **Colab cannot enable GPU from code**—you must choose it once in the menu:

1. Click **Runtime** (or the **▶ Connect** dropdown) → **Change runtime type**  
2. Set **Hardware accelerator** to **T4 GPU** → **Save**  
3. Re-run the notebook from the top  

After that, the check below will report "GPU OK".


In [ ]:
# Check GPU (Colab cannot enable GPU from code—you must set it in the menu once)
try:
    gpu_name = __import__("subprocess").check_output(
        ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], text=True
    ).strip().split("\n")[0]
except Exception:
    gpu_name = ""
if not gpu_name or "T4" not in gpu_name.upper():
    print("WARNING: T4 GPU not detected.")
    print("  → Runtime → Change runtime type → Hardware accelerator: T4 GPU → Save")
    print("  → Then re-run the notebook from the top.")
else:
    print("GPU OK:", gpu_name)

### 2. Install the GeoAI package

Run the next cell to install the `geoai-py` package. If it is already installed, you can skip the cell or run it anyway.


In [ ]:
%pip install geoai-py

### 3. Import the packages

Import **geoai** and all other packages used in this lab (leafmap, numpy, matplotlib, rasterio, geopandas, ipywidgets, etc.) so they are available for the rest of the notebook.


In [ ]:
import geoai
import os
import shutil
import gc
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import rasterio
from rasterio.features import rasterize
import geopandas as gpd
import leafmap
from ipywidgets import HBox, IntSlider, VBox, Output, interactive
from IPython.display import display, HTML

### 4. Mount Google Drive and create the lab folder

All data, tiles, models, and outputs will be saved under **My Drive → ABT182_GeoAI** so you can reuse them and avoid re-downloading.


In [ ]:
from google.colab import drive

drive.mount("/content/drive")

# Base folder in your Google Drive
BASE_DIR = "/content/drive/MyDrive/ABT182_GeoAI"
os.makedirs(f"{BASE_DIR}/data/train", exist_ok=True)
os.makedirs(f"{BASE_DIR}/data/test", exist_ok=True)
os.makedirs(f"{BASE_DIR}/tiles", exist_ok=True)
os.makedirs(f"{BASE_DIR}/models", exist_ok=True)
os.makedirs(f"{BASE_DIR}/outputs", exist_ok=True)
print(f"Created: {BASE_DIR}")
print("  data/train, data/test, tiles, models, outputs")

### 5. Colab Pro (Yes/No)

Run the next cell and answer the prompt: **Are you using Google Colab Pro? (Yes/No)**. Type `Yes` or `No` and press Enter. Settings will adjust automatically: Pro uses larger batches and training visuals; free uses batch size 1 and fewer windows to avoid out-of-memory errors.


In [ ]:
# Answer the prompt: Are you using Colab Pro? (Yes/No)
reply = input("Are you using Google Colab Pro? (Yes/No): ").strip().lower()
COLAB_PRO = reply in ("yes", "y")

USE_LOW_RAM = not COLAB_PRO

if USE_LOW_RAM:
    TILE_SIZE = 256
    STRIDE = 256
    BATCH_SIZE = 1
    WINDOW_SIZE = 256
    OVERLAP = 64
    NUM_EPOCHS = 5
    VISUALIZE_TRAINING = False
else:
    TILE_SIZE = 512
    STRIDE = 256
    BATCH_SIZE = 4
    WINDOW_SIZE = 512
    OVERLAP = 256
    NUM_EPOCHS = 10
    VISUALIZE_TRAINING = True
print("COLAB_PRO =", COLAB_PRO, "| USE_LOW_RAM =", USE_LOW_RAM)

<a name="section1"></a>
## **Exercise 1** – Download sample data

**What this section does:** Download the training raster, vector, and test raster from the URLs below, then copy them to your Drive. **What to fill in:** Replace each **XXXXX** in the next code cell: the first three **XXXXX** are `train_raster_url`, `train_vector_url`, `test_raster_url`; the three `shutil.copy` pairs are `(train_file, train_raster_path)`, `(vector_file, train_vector_path)`, `(test_file, test_raster_path)`.

In [ ]:
train_raster_url = (
    "https://huggingface.co/datasets/giswqs/geospatial/resolve/main/naip_rgb_train.tif"
)
train_vector_url = "https://huggingface.co/datasets/giswqs/geospatial/resolve/main/naip_train_buildings.geojson"
test_raster_url = (
    "https://huggingface.co/datasets/giswqs/geospatial/resolve/main/naip_test.tif"
)

In [ ]:
# Download and copy to Drive. Replace each XXXXX as explained in the section above.
train_file = geoai.download_file(XXXXX)  
vector_file = geoai.download_file(XXXXX)  
test_file = geoai.download_file(XXXXX)     

train_raster_path = f"{BASE_DIR}/data/train/naip_rgb_train.tif"
train_vector_path = f"{BASE_DIR}/data/train/naip_train_buildings.geojson"
test_raster_path = f"{BASE_DIR}/data/test/naip_test.tif"

shutil.copy(XXXXX, XXXXX)  
shutil.copy(XXXXX, XXXXX)  
shutil.copy(XXXXX, XXXXX)  
print("Saved to Drive:", BASE_DIR)

<a name="section2"></a>
## **Exercise 2** – Visualize sample data

**What this section does:** Inspect the training raster metadata, then visualize what you downloaded in Exercise 1 in separate maps: (1) training raster only (true color), (2) building footprints (vector) only, (3) training raster with buildings on top, (4) test raster only (true color), (5) and (6) make a **false color composite** (NIR, R, G) for train and test—NAIP has 4 bands (R, G, B, NIR); use bands `[4, 3, 2]` to display NIR as red, blue as green, green as blue (vegetation appears red). You can also open the downloaded GeoTIFF and GeoJSON files in **ArcGIS** to explore them there.

**What to fill in:** In each map cell that uses a raster, replace `XXXXX` with the band list: `[1, 2, 3]` for true color (RGB), or `[4, 3, 2]` for false color (NIR-B-G).

In [ ]:
geoai.get_raster_info(train_raster_path)

In [ ]:
# 1. Training raster only (NAIP). FILL: bands for RGB, e.g. [1, 2, 3]
m = leafmap.Map()
m.add_raster(train_raster_path, layer_name="Train NAIP", bands=XXXXX)
m

In [ ]:
# 2. Building footprints (vector) only – zoom to layer extent
m = leafmap.Map()
m.add_geojson(train_vector_path, layer_name="Buildings", style={"stroke": True, "color": "#ff0000", "weight": 2, "fill": False, "fillOpacity": 0})
gdf_bounds = gpd.read_file(train_vector_path)
if len(gdf_bounds) > 0:
    b = gdf_bounds.total_bounds
    m.fit_bounds([[b[1], b[0]], [b[3], b[2]]])
m

In [ ]:
# 3. Training raster with buildings on top. FILL: bands for RGB, e.g. [1, 2, 3]
m = leafmap.Map()
m.add_raster(train_raster_path, layer_name="NAIP", bands=XXXXX)
m.add_geojson(train_vector_path, layer_name="Buildings", style={"stroke": True, "color": "#ff0000", "weight": 2, "fill": False, "fillOpacity": 0})
m

In [ ]:
# 4. Test raster only. FILL: same bands for RGB, e.g. [1, 2, 3]
m = leafmap.Map()
m.add_raster(test_raster_path, layer_name="Test NAIP", bands=XXXXX)
m

In [ ]:
# 5. False color composite – Test (NIR, B, G). FILL: same bands [4, 3, 2]
m = leafmap.Map()
m.add_raster(test_raster_path, layer_name="Test NAIP (NIR-R-G)", bands=XXXXX)
m

<a name="section3"></a>
## **Exercise 3** – Create training data

**What this section does:** Create training tiles from the raster and vector; the train image is cut into smaller patches. `out_folder` path is given. After the export, you'll print the number of tiles and show a few sample tile images.

**What to fill in:** Replace only: `XXXXX` → `train_raster_path`, `XXXXX` → `train_vector_path`, `XXXXX` → `TILE_SIZE`, `XXXXX` → `STRIDE`.

In [ ]:
out_folder = f"{BASE_DIR}/tiles/buildings_instance"
tiles = geoai.export_geotiff_tiles(
    in_raster=XXXXX,
    out_folder=out_folder,
    in_class_data=XXXXX,
    tile_size=XXXXX,
    stride=XXXXX,
    buffer_radius=0,
)

In [ ]:
# After creating tiles: print how many tiles and show a few (train image was cut into smaller patches)
images_dir = f"{out_folder}/images"
tile_files = sorted([f for f in os.listdir(images_dir) if f.lower().endswith(('.tif', '.tiff', '.png'))])
n_tiles = len(tile_files)
print(f"Number of image tiles created: {n_tiles}")
print(f"Tile size (pixels): {TILE_SIZE} x {TILE_SIZE}")
print(f"Stride: {STRIDE}")
print(f"First few tiles: {tile_files[:5]}")

In [ ]:
# Display sample tiles: 4 per row, up to 40 tiles. Each pair of rows = image only, then same tiles with label overlay.
labels_dir = f"{out_folder}/labels"
n_cols = 4
n_show = min(40, n_tiles)
n_blocks = (n_show + n_cols - 1) // n_cols
n_rows = 2 * n_blocks
fig, axes = plt.subplots(n_rows, n_cols, figsize=(3 * n_cols, 2.5 * n_rows))
if n_blocks == 1:
    axes = axes.reshape(2, n_cols)
for block in range(n_blocks):
    for col in range(n_cols):
        idx = block * n_cols + col
        if idx >= n_show:
            axes[2 * block, col].axis("off")
            axes[2 * block + 1, col].axis("off")
            continue
        path = os.path.join(images_dir, tile_files[idx])
        with rasterio.open(path) as src:
            img = src.read()
            if img.shape[0] >= 3:
                rgb = np.transpose(img[:3], (1, 2, 0))
            else:
                rgb = np.transpose(img, (1, 2, 0))
            if rgb.max() > 1:
                rgb = np.clip(rgb / 255.0, 0, 1)
            else:
                rgb = np.clip(rgb, 0, 1)
        axes[2 * block, col].imshow(rgb)
        axes[2 * block, col].set_title(tile_files[idx][:20])
        axes[2 * block, col].axis("off")
        label_path = os.path.join(labels_dir, tile_files[idx])
        if os.path.isfile(label_path):
            with rasterio.open(label_path) as src_l:
                mask = src_l.read()
                mask = np.squeeze(mask)
                mask_binary = (mask > 0).astype(float)
            overlay = rgb.copy()
            overlay[mask_binary > 0, 0] = np.clip(overlay[mask_binary > 0, 0] + 0.5, 0, 1)
            overlay[mask_binary > 0, 1] = overlay[mask_binary > 0, 1] * 0.5
            overlay[mask_binary > 0, 2] = overlay[mask_binary > 0, 2] * 0.5
            axes[2 * block + 1, col].imshow(overlay)
        else:
            axes[2 * block + 1, col].imshow(rgb)
        axes[2 * block + 1, col].set_title(f"{tile_files[idx][:15]} + labels")
        axes[2 * block + 1, col].axis("off")
axes[0, 0].set_ylabel("Image only", fontsize=10)
axes[1, 0].set_ylabel("Image + labels", fontsize=10)
plt.tight_layout()
plt.show()

<a name="section4"></a>
## **Exercise 4** – Train instance segmentation model

**What this section does:** Train Mask R-CNN on the tiles. You'll print the training configuration, run training with verbose output (loss and metrics per epoch), then plot the **loss** and **accuracy** curves for train and validation. Paths under `out_folder` are used.

**What to fill in:** Replace only: `XXXXX` → `f"{out_folder}/images"`, `XXXXX` → `f"{out_folder}/labels"`, `XXXXX` → `f"{out_folder}/instance_models"`.

In [ ]:
# Print training configuration before starting
print("Training configuration:")
print(f"  images_dir     = {out_folder}/images")
print(f"  labels_dir     = {out_folder}/labels")
print(f"  output_dir     = {out_folder}/instance_models")
print(f"  num_classes    = 2 (background + building)")
print(f"  num_epochs     = {NUM_EPOCHS}")
print(f"  batch_size     = {BATCH_SIZE}")
print(f"  learning_rate  = 0.005")
print(f"  val_split      = 0.2 (20% validation)")
print(f"  visualize      = {VISUALIZE_TRAINING}")
print("Training will print loss and metrics per epoch (verbose=True).")

In [ ]:
geoai.train_instance_segmentation_model(
    images_dir=XXXXX,
    labels_dir=XXXXX,
    output_dir=XXXXX,
    num_classes=2,
    num_channels=3,
    batch_size=BATCH_SIZE,
    num_epochs=NUM_EPOCHS,
    learning_rate=0.005,
    val_split=0.2,
    visualize=VISUALIZE_TRAINING,
    verbose=True,
)

In [ ]:
# Plot loss and accuracy (train and validation) right after training
history_path = f"{out_folder}/instance_models/training_history.pth"
print("Training history saved to:", history_path)
if os.path.isfile(history_path):
    geoai.plot_performance_metrics(history_path=history_path, figsize=(14, 5), verbose=True)
    import matplotlib.pyplot as plt
    for ax in plt.gcf().axes:
        if ax.get_legend() is not None:
            ax.get_legend().set_visible(False)
    plt.draw()
    print("Best model saved to:", f"{out_folder}/instance_models/best_model.pth")
    print("Files in output_dir:", os.listdir(f"{out_folder}/instance_models"))
else:
    print("History file not found yet. Run the training cell above first.")


<a name="section5"></a>
## **Exercise 5** – Run inference

**What this section does:** Run the trained model on the test image; the output is a mask raster saved to `masks_path`. You'll print the inference parameters, run the model, then print the output raster metadata. The mask will be visualized in Exercise 7. Output mask path is given.

**What to fill in:** In the inference call only, replace `XXXXX` → `test_raster_path`, `XXXXX` → `masks_path`, `XXXXX` → `model_path`.

In [ ]:
# Paths given (no fill needed)
masks_path = f"{BASE_DIR}/outputs/naip_test_instance_prediction.tif"
model_path = f"{out_folder}/instance_models/best_model.pth"

In [ ]:
# Print inference parameters before running
print("Inference configuration:")
print(f"  input_path    = {test_raster_path}")
print(f"  output_path   = {masks_path}")
print(f"  model_path    = {model_path}")
print(f"  window_size   = {WINDOW_SIZE}")
print(f"  overlap       = {OVERLAP}")
print(f"  batch_size    = {BATCH_SIZE}")
print(f"  confidence_threshold = 0.5")
print("Running inference...")

In [ ]:
# Run instance segmentation on the test image. Replace each XXXXX with test_raster_path, masks_path, model_path.
geoai.instance_segmentation(
    input_path=XXXXX,
    output_path=XXXXX,
    model_path=XXXXX,
    num_classes=2,
    num_channels=3,
    window_size=WINDOW_SIZE,
    overlap=OVERLAP,
    confidence_threshold=0.5,
    batch_size=BATCH_SIZE,
)
gc.collect()

In [ ]:
# After inference: print output path and raster metadata. Visualization is in Exercise 7.
print("Inference complete. Predicted mask saved to:", masks_path)
print("\nOutput mask raster info:")
geoai.get_raster_info(masks_path)

<a name="section6"></a>
## **Exercise 6** – Vectorize masks & add geometric properties

**What this section does:** Convert the mask raster to polygons and add area (m²), perimeter, etc. Path for output GeoJSON is given.

**What to fill in:** In the first cell, replace `XXXXX` with the variable holding the mask raster path (e.g. `masks_path`). In the second, replace `XXXXX` with the GeoDataFrame from the previous step (e.g. `gdf`).

In [ ]:
output_vector_path = f"{BASE_DIR}/outputs/naip_test_instance_prediction.geojson"
gdf = geoai.orthogonalize(XXXXX, output_vector_path, epsilon=2)

In [ ]:
gdf_props = geoai.add_geometric_properties(XXXXX, area_unit="m2", length_unit="m")

<a name="section7"></a>
## **Exercise 7** – Visualize results on test set (box plot, filter by area)

**What this section does:** Show the predicted mask over the test NAIP image, buildings colored by area, a **histogram** of areas, then **filter by minimum area** (e.g. 50 m²) and a **box plot by size range** to see which area ranges are more probable. Run the CSS cell first so popup text is black.

**What to fill in:** In the map cells, replace: `XXXXX` → e.g. `[1, 2, 3]` for RGB; `XXXXX` → e.g. `"Predicted masks"`; `XXXXX` → e.g. `"YlOrRd"` for area colors; `XXXXX` → e.g. `"Quantiles"`. For the histogram and box plot, fill `XXXXX`, `XXXXX`, and `XXXXX`.

In [ ]:
# Run this first so popup/tooltip text is black (readable on white background)
display(HTML("""
<style>
.leaflet-popup-content-wrapper, .leaflet-popup-content { color: #000 !important; }
.leaflet-tooltip { color: #000 !important; }
</style>
"""))

In [ ]:
# NAIP + masks. FILL: bands (e.g. [1,2,3]), layer_name (e.g. "Predicted masks")
m = leafmap.Map()
m.add_raster(test_raster_path, layer_name="NAIP", bands=XXXXX)
m.add_raster(masks_path, layer_name=XXXXX, cmap="tab20", nodata=0)
m

In [ ]:
# Compare with/without masks: two maps side by side (split_map does not work with local rasters)
m_left = leafmap.Map()
m_left.add_raster(test_raster_path, layer_name="NAIP only", bands=XXXXX)
m_right = leafmap.Map()
m_right.add_raster(test_raster_path, layer_name="NAIP", bands=XXXXX)
m_right.add_raster(masks_path, layer_name=XXXXX, cmap="tab20", nodata=0)
display(HBox([m_left, m_right]))

In [ ]:
# Buildings colored by area. FILL: bands, scheme (e.g. "Quantiles"), cmap (e.g. "YlOrRd")
m = leafmap.Map()
m.add_raster(test_raster_path, layer_name="NAIP", bands=XXXXX)
m.add_data(gdf_props, column="area_m2", scheme=XXXXX, cmap=XXXXX, legend_title="Area (m²)")
m

**Histogram of building areas:** Replace **XXXXX** in `bins=XXXXX` with a number (e.g. `25`).

In [ ]:
# Histogram of building areas (area_m2). Replace XXXXX with number of bins (e.g. 25).
plt.figure(figsize=(8, 4))
plt.hist(gdf_props["area_m2"], bins=XXXXX, color="steelblue", edgecolor="white")
plt.xlabel("Area (m²)"); plt.ylabel("Count"); plt.title("Distribution of predicted building areas")
plt.tight_layout(); plt.show()

**Filter by area:** Keep only buildings above a minimum area (e.g. 50 m²), then visualize filtered buildings. **Box plot by size range:** Below, group buildings into size ranges to see which area ranges are more probable.

**What to fill in:** Replace `XXXXX` with a number in m² (e.g. `50`).

In [ ]:
gdf_filtered = gdf_props[(gdf_props["area_m2"] > XXXXX)]

In [ ]:
# NAIP + filtered buildings (paths and params given)
m = leafmap.Map()
m.add_raster(test_raster_path, layer_name="NAIP", bands=[1, 2, 3])
m.add_data(gdf_filtered, column="area_m2", scheme="Quantiles", cmap="YlOrRd", legend_title="Area (m²)")
m

## Compare predictions with imagery

In [ ]:
# Compare view (colored by area_m2)
m = leafmap.Map()
m.add_raster(test_raster_path, layer_name="NAIP", bands=[1, 2, 3])
m.add_data(gdf_filtered, column="area_m2", scheme="Quantiles", cmap="RdYlBu_r", legend_title="Area (m²)")
m

**Box plot by size range:** Group buildings into size ranges and plot a box plot. **What to fill in:** e.g. define bins and labels for `pd.cut()` (e.g. bins=[0, 100, 250, 500, 2000], labels=["0-100", "100-250", "250-500", "500+"]).

In [ ]:
# Box plot by size range. Replace the two XXXXX: bins (e.g. [0, 100, 250, 500, 2000]) and labels (e.g. ["0-100", "100-250", "250-500", "500+"]).
gdf_f = gdf_filtered.copy()
gdf_f["size_range"] = pd.cut(gdf_f["area_m2"], bins=XXXXX, labels=XXXXX)
fig, ax = plt.subplots(figsize=(8, 4))
labels = XXXXX  
data = [gdf_f[gdf_f["size_range"] == lb]["area_m2"].values for lb in labels]
bp = ax.boxplot(data, labels=labels, patch_artist=True)
colors = plt.cm.Blues(np.linspace(0.35, 0.85, len(labels)))
for patch, color in zip(bp["boxes"], colors):
    patch.set_facecolor(color)
ax.set_xlabel("Area range (m²)"); ax.set_ylabel("Area (m²)"); ax.set_title("Building areas by size range")
plt.tight_layout(); plt.show()

<a name="section8"></a>
## **Exercise 8** – Compare predictions on train set (actual vs predicted, metrics)

**What we do here:** We do not have ground-truth labels for the test set, so we compare predictions with actual labels on the **train set**. Run the same trained model on the train raster to get predicted masks, vectorize them, then compare **actual labels (left)** with **predicted (right)** in a side-by-side map and compute performance scores (IoU, precision, recall, F1).

**What to fill in:** In the inference cell use `input_path=train_raster_path`, `output_path=train_masks_path`, `model_path=model_path`. Other cells use the variable names from the hints.

In [ ]:
# Run inference on the TRAIN raster (same model). FILL: input_path=train_raster_path, output_path=train_masks_path, model_path=model_path
train_masks_path = f"{BASE_DIR}/outputs/train_instance_masks.tif"
geoai.instance_segmentation(
    input_path=train_raster_path,
    output_path=train_masks_path,
    model_path=model_path,
    num_classes=2,
    num_channels=3,
    window_size=WINDOW_SIZE,
    overlap=OVERLAP,
    batch_size=BATCH_SIZE,
    confidence_threshold=0.5,
)
gc.collect()
print("Train set inference complete. Masks saved to:", train_masks_path)

In [ ]:
# Vectorize train predictions and add geometric properties
train_vector_pred_path = f"{BASE_DIR}/outputs/train_instance_prediction.geojson"
gdf_train_pred = geoai.orthogonalize(train_masks_path, train_vector_pred_path, epsilon=2)
gdf_train_pred = geoai.add_geometric_properties(gdf_train_pred, area_unit="m2", length_unit="m")
print("Predicted buildings on train set:", len(gdf_train_pred))

In [ ]:
# Load actual (ground-truth) building footprints for the train area
gdf_actual = gpd.read_file(train_vector_path)
if gdf_actual.crs is None:
    gdf_actual.set_crs(epsg=4326, inplace=True)
print("Actual buildings (train set):", len(gdf_actual))

**Slider: Actual (left) vs Predicted (right)** — Use the slider below to switch between viewing only **actual labels** (ground truth) and only **predicted** buildings on the train set.

In [ ]:
# Slider: view Actual (0) or Predicted (1) — build one map on demand to save RAM
out = Output()
def show_map(view):
    out.clear_output(wait=True)
    with out:
        m = leafmap.Map()
        m.add_raster(train_raster_path, layer_name="NAIP", bands=[1, 2, 3])
        if view == 0:
            m.add_geojson(gdf_actual.to_json(), layer_name="Actual buildings", style={"stroke": True, "color": "lime", "weight": 2, "fill": False})
        else:
            m.add_data(gdf_train_pred, column="area_m2", scheme="Quantiles", cmap="YlOrRd", legend_title="Predicted area (m²)")
        m.add_layer_control()
        display(m)
slider = IntSlider(min=0, max=1, step=1, value=0, description="Actual (0) / Predicted (1)")
w = interactive(show_map, view=slider)
display(VBox([w, out]))
# Move the slider to 0 (Actual) or 1 (Predicted) to build and show the map (saves RAM until you need it)


In [ ]:
# Rasterize actual and predicted to same grid; compute IoU, precision, recall, F1 (pixel-level)
with rasterio.open(train_raster_path) as src:
    transform = src.transform
    width, height = src.width, src.height
    crs = src.crs

gdf_actual_raster = gdf_actual.to_crs(crs)
gdf_train_pred_raster = gdf_train_pred.to_crs(crs)

actual_binary = rasterize(
    [(geom, 1) for geom in gdf_actual_raster.geometry],
    out_shape=(height, width),
    transform=transform,
    fill=0,
    dtype=np.uint8,
)
pred_binary = rasterize(
    [(geom, 1) for geom in gdf_train_pred_raster.geometry],
    out_shape=(height, width),
    transform=transform,
    fill=0,
    dtype=np.uint8,
)

intersection = np.logical_and(actual_binary == 1, pred_binary == 1).sum()
union = np.logical_or(actual_binary == 1, pred_binary == 1).sum()
actual_area = (actual_binary == 1).sum()
pred_area = (pred_binary == 1).sum()

iou = intersection / union if union > 0 else 0.0
precision = intersection / pred_area if pred_area > 0 else 0.0
recall = intersection / actual_area if actual_area > 0 else 0.0
f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0

print("Performance on train set (pixel-level):")
print(f"  IoU:       {iou:.4f}")
print(f"  Precision: {precision:.4f}")
print(f"  Recall:    {recall:.4f}")
print(f"  F1:        {f1:.4f}")
print(f"  Actual building pixels: {actual_area}")
print(f"  Predicted building pixels: {pred_area}")

<a name="section9"></a>
## **Exercise 9** – Model performance

**What this section does:** Plot training/validation loss and accuracy from the saved history. Path is given below; just run the cell.

In [ ]:
# Training and validation curves
history_path = f"{out_folder}/instance_models/training_history.pth"
if os.path.isfile(history_path):
    geoai.plot_performance_metrics(history_path=history_path, figsize=(15, 5), verbose=True)
else:
    print("History file not found. Run the training cell first.")

---
## **Credits**

This lab uses the **[GeoAI](https://opengeoai.org/)** Python package. We thank **Dr. Qiusheng Wu** for creating GeoAI and for the examples that inspired this tutorial.

For more information, documentation, and examples, visit: **https://opengeoai.org/**

<a name="section_submit"></a>
## **How to Submit Lab 9**

Once you have finished the exercises, follow the steps below to submit your assignment.

### Step 1: Run the Entire Notebook
- Run all code cells to make sure your notebook works correctly and displays all results.
- Go to the **Runtime** tab and click **Run all**.

### Step 2: Check Your Work
- Make sure you followed all instructions.
- Confirm that you completed the Academic Integrity cell and that all outputs look correct.

### Step 3: Rename and Save the Notebook
- Click on the notebook name at the **top left** of the page.
- Rename the file by replacing the default name with your own (e.g., **lastname_firstname_lab9.ipynb**).
- Save the notebook after renaming.

### Step 4: Create PDF Version
- Creating a PDF version of your notebook is **required**.
- **File** → **Print** → set **Destination** to **Save as PDF**, or **File** → **Download** → **Download .pdf**.

### Step 5: Submit Both Files to Canvas
- **Upload BOTH files** to the **Canvas** assignment for Lab 9:
  1. The renamed notebook (`.ipynb`) – **REQUIRED**
  2. The PDF (`.pdf`) – **REQUIRED**
- Submit before **March 11, 11:59 PM 2026**.

### Use of Generative AI Tools (GenAI Policy)
- This course **discourages the use of generative AI tools** (such as ChatGPT) for completing assignments, so you can develop your own skills.
- If you **do use a generative AI tool**, you must **clearly disclose** it in your notebook (which tool and how you used it).
- Failure to disclose may be considered a violation of course academic integrity policies.